# Best scenario with net capture target

This notebook finds the cheapest feasible scenario that still achieves at least `35.8 MtCO2/yr` net capture, then plots the winning simulation.

In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import src.assumptions as A
from src.data import renewable_capacity_factors
from src.demand_model import DemandMode, predicted_demand
from src.power_system import PowerSystem
from src.power_system_no_h2 import PowerSystemNoH2
from src.supply_model import daily_renewables_capacity
from src.units import Units as U

import src.matplotlib_style  # noqa: F401

In [ ]:
TARGET_NET_CAPTURE_MT = 35.8
DAC_MAX_GW = 50.0
DAC_TOL_MT = 0.1

ENABLE_IMPORTS = False
CAPACITY_FACTORS_SOURCE = "era5_2024"
DEMAND_MODE = DemandMode.SEASONAL
REN_RANGE = range(140, 351, 10)

HYDROGEN_STORAGE_TWH = A.HydrogenStorage.CavernStorage.Capacity
ELECTROLYSER_GW = A.HydrogenStorage.Electrolysis.Power
HYDROGEN_GEN_GW = A.HydrogenStorage.Generation.Power

demand_df = predicted_demand(mode=DEMAND_MODE, historical="era5", filter_ldz=True)
daily_cf = renewable_capacity_factors.get_renewable_capacity_factors(
    source=CAPACITY_FACTORS_SOURCE,
    resample="D",
)


def build_net_supply_df(ren_gw: int) -> pd.DataFrame:
    supply = daily_renewables_capacity(ren_gw * U.GW, daily_cf)
    common_idx = supply.index.intersection(demand_df.index)
    supply = supply.reindex(common_idx)
    demand = demand_df.reindex(common_idx)
    net_supply_df = (supply - demand["demand"]).to_frame(name=f"S-D(TWh),Ren={ren_gw}GW")
    net_supply_df = net_supply_df.reset_index()
    net_supply_df = net_supply_df.rename(columns={net_supply_df.columns[0]: "index"})
    return net_supply_df

In [ ]:
def make_system(mode: str, ren_gw: int, dac_gw: float):
    if mode == "with_h2":
        return PowerSystem(
            renewable_capacity=ren_gw * U.GW,
            hydrogen_storage_capacity=HYDROGEN_STORAGE_TWH,
            electrolyser_power=ELECTROLYSER_GW,
            dac_capacity=dac_gw * U.GW,
            hydrogen_generation_power=HYDROGEN_GEN_GW,
            enable_imports=ENABLE_IMPORTS,
            capacity_factors_source=CAPACITY_FACTORS_SOURCE,
            enable_gas_ltdac=True,
        )
    if mode == "no_h2":
        return PowerSystemNoH2(
            renewable_capacity=ren_gw * U.GW,
            dac_capacity=dac_gw * U.GW,
            enable_imports=ENABLE_IMPORTS,
            capacity_factors_source=CAPACITY_FACTORS_SOURCE,
        )
    raise ValueError(f"Unsupported mode: {mode}")


def run_scenario(mode: str, ren_gw: int, dac_gw: float) -> tuple[object, pd.DataFrame | None, dict | None]:
    net_supply_df = build_net_supply_df(ren_gw)
    system = make_system(mode, ren_gw, dac_gw)
    sim_df = system.run_simulation(net_supply_df)
    analysis = system.analyze_simulation_results(sim_df)
    return system, sim_df, analysis


def net_capture_mt(mode: str, ren_gw: int, dac_gw: float) -> float | None:
    _, _, analysis = run_scenario(mode, ren_gw, dac_gw)
    if analysis is None:
        return None
    return analysis["annual_net_capture"].to(U.Mt).magnitude


def find_min_dac_gw(
    *,
    mode: str,
    ren_gw: int,
    target_mt: float,
    dac_min: float = 0.0,
    dac_max: float = DAC_MAX_GW,
    tol_mt: float = DAC_TOL_MT,
    max_iter: int = 30,
) -> dict[str, float | str]:
    low = dac_min
    high = dac_max

    low_val = net_capture_mt(mode, ren_gw, low)
    high_val = net_capture_mt(mode, ren_gw, high)

    if low_val is None or high_val is None:
        return {
            "mode": mode,
            "ren_gw": ren_gw,
            "dac_gw": math.nan,
            "net_capture_mt": math.nan,
            "status": "simulation_failed",
        }

    while high_val < target_mt and high < 400:
        low, low_val = high, high_val
        high *= 2
        high_val = net_capture_mt(mode, ren_gw, high)
        if high_val is None:
            return {
                "mode": mode,
                "ren_gw": ren_gw,
                "dac_gw": math.nan,
                "net_capture_mt": math.nan,
                "status": "simulation_failed",
            }

    if high_val < target_mt:
        return {
            "mode": mode,
            "ren_gw": ren_gw,
            "dac_gw": math.nan,
            "net_capture_mt": high_val,
            "status": "target_unreachable",
        }

    mid = high
    mid_val = high_val
    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        mid_val = net_capture_mt(mode, ren_gw, mid)
        if mid_val is None:
            return {
                "mode": mode,
                "ren_gw": ren_gw,
                "dac_gw": math.nan,
                "net_capture_mt": math.nan,
                "status": "simulation_failed",
            }
        if abs(mid_val - target_mt) <= tol_mt:
            return {
                "mode": mode,
                "ren_gw": ren_gw,
                "dac_gw": mid,
                "net_capture_mt": mid_val,
                "status": "ok",
            }
        if mid_val < target_mt:
            low, low_val = mid, mid_val
        else:
            high, high_val = mid, mid_val

    return {
        "mode": mode,
        "ren_gw": ren_gw,
        "dac_gw": mid,
        "net_capture_mt": mid_val,
        "status": "max_iter",
    }


def evaluate_targeted_scenario(mode: str, ren_gw: int) -> dict[str, float | str | None]:
    target_result = find_min_dac_gw(mode=mode, ren_gw=ren_gw, target_mt=TARGET_NET_CAPTURE_MT)
    if target_result["status"] not in {"ok", "max_iter"}:
        return target_result

    dac_gw = float(target_result["dac_gw"])
    system, sim_df, analysis = run_scenario(mode, ren_gw, dac_gw)
    if analysis is None:
        return {
            "mode": mode,
            "ren_gw": ren_gw,
            "dac_gw": dac_gw,
            "net_capture_mt": math.nan,
            "status": "simulation_failed",
        }

    power_cost = system.calculate_power_system_cost(sim_df).to(U.GBP).magnitude
    dac_cost = analysis["annual_dac_total_cost"].to(U.GBP).magnitude
    total_cost = power_cost + dac_cost
    total_cost_per_mwh = (total_cost * U.GBP / A.EnergyDemand2050).to(U.GBP / U.MWh).magnitude

    return {
        "mode": mode,
        "ren_gw": ren_gw,
        "dac_gw": dac_gw,
        "status": str(target_result["status"]),
        "net_capture_mt": analysis["annual_net_capture"].to(U.Mt).magnitude,
        "power_cost_gbp": power_cost,
        "dac_cost_gbp": dac_cost,
        "total_cost_gbp": total_cost,
        "total_cost_per_mwh": total_cost_per_mwh,
        "annual_gas_twh": analysis["annual_gas_ccs_energy"].to(U.TWh).magnitude,
        "annual_dac_twh": analysis["annual_dac_energy"].to(U.TWh).magnitude,
        "annual_grid_dac_capture_mt": analysis["annual_co2_removals"].to(U.Mt).magnitude,
        "annual_gas_dac_capture_mt": analysis["annual_gas_dac_capture"].to(U.Mt).magnitude,
        "annual_gas_emissions_mt": analysis["annual_gas_emissions"].to(U.Mt).magnitude,
        "curtailed_twh": analysis["curtailed_energy"].to(U.TWh).magnitude,
        "min_medium_twh": analysis["minimum_medium_storage"].to(U.TWh).magnitude,
        "min_h2_twh": analysis.get("minimum_hydrogen_storage", math.nan).to(U.TWh).magnitude
        if mode == "with_h2"
        else math.nan,
    }

In [ ]:
rows: list[dict[str, float | str | None]] = []
for ren_gw in REN_RANGE:
    for mode in ["with_h2", "no_h2"]:
        rows.append(evaluate_targeted_scenario(mode, ren_gw))

results = pd.DataFrame(rows)
feasible = results[
    results["status"].isin(["ok", "max_iter"]) & (results["net_capture_mt"] >= TARGET_NET_CAPTURE_MT - DAC_TOL_MT)
].copy()
best = feasible.sort_values("total_cost_gbp").iloc[0]

summary_cols = [
    "mode",
    "ren_gw",
    "dac_gw",
    "net_capture_mt",
    "total_cost_gbp",
    "total_cost_per_mwh",
    "annual_grid_dac_capture_mt",
    "annual_gas_dac_capture_mt",
    "annual_gas_emissions_mt",
    "annual_gas_twh",
    "curtailed_twh",
]

display(feasible.sort_values("total_cost_gbp")[summary_cols].head(10).round(3))
display(best[summary_cols].to_frame(name="best_scenario"))

In [ ]:
best_mode = str(best["mode"])
best_ren_gw = int(best["ren_gw"])
best_dac_gw = float(best["dac_gw"])

best_system, best_sim_df, best_analysis = run_scenario(best_mode, best_ren_gw, best_dac_gw)
best_label = (
    f"best_{best_mode}_{best_ren_gw}GW_dac_{best_dac_gw:.2f}GW_"
    f"net_{best_analysis['annual_net_capture'].to(U.Mt).magnitude:.2f}Mt"
)

print(f"Best mode: {best_mode}")
print(f"Renewables: {best_ren_gw} GW")
print(f"DAC capacity: {best_dac_gw:.3f} GW")
print(f"Net CO2 capture: {best_analysis['annual_net_capture']:~0.2f}")
print(f"Total annual cost: {(best['total_cost_gbp'] * U.GBP):~0.2e}")
print(f"Average cost: {(best['total_cost_per_mwh'] * U.GBP / U.MWh):~0.2f}")

best_system.plot_simulation_results(best_sim_df, best_analysis, demand_mode=best_label, fname=None)
plt.show()